## Ideas / Plan of Action

### Training set optimization

#### Basic approaches

Try:

- CDmean, PEVmean, Random
- GBLUP, NN, GNN

#### More advanced approaches

- Data shapley
- Tracin
- GLISTER
- GRAD-MATCH
---
- CDMean on gblup, --> Test also on nn?
- Pruning with EL2N/Tracin --> Test also on gblup?
- Compare with random
- Analysis of selected sets (which individuals / islands are selected?)

### Tracin

- Removal curves (0, 10, 20 \% etc.)
- Weighted retraining (weighted by the tracin score)
- Iterative pruning (recompute tracin scores after each pruning step)



# Chat ideer:

This is a *really* strong direction.

Switching to ridge is a smart move because (i) it’s a standard genomic prediction workhorse (RR-BLUP / kernel ridge ideas) and (ii) it makes Shapley-style training-set optimization computationally realistic. ([acsess.onlinelibrary.wiley.com][1])
And your Lovund example is exactly the kind of “Figure 3(b)” story you wanted: **remove low-value sources → test corr improves a lot**, with proper cal/test separation.

Here’s what I’d do next, both for *this* island-Shapley track and for the thesis “training set optimization” big picture.

---

## 1) Lock the result in properly (before scaling)

Do these three checks for 2–3 target islands (including Lovund):

### A) Stability vs permutations

Run TMC with e.g. **10, 20, 50, 100** permutations and track:

* rank correlation of (\phi) (or top-k overlap) vs permutation count
* the add/remove curves on target_test

If 50 is already stable, great; if not, you’ll know what you need.

### B) Stability vs calibration fraction

Repeat with `cal_frac ∈ {0.05, 0.10, 0.20, 0.30}` (fixed target_test).
This is thesis gold: “how many labeled target individuals are needed before guided selection helps?”

### C) Add the baseline you suggested: **size-matched random individuals**

For each removal level, match the **same total number of training individuals** by sampling randomly from the whole source pool (optionally stratified by island). This isolates “good islands” vs “just fewer data”.

---

## 2) Scale up in a way that produces thesis-ready figures

### A) All target islands (LOIO outer loop)

For each target island:

* compute island Shapley values (on target_cal)
* evaluate curves on target_test
  Store:
* best improvement in corr and MSE vs full-data baseline
* the “optimal #islands remaining” per target

Then make two summary plots:

* distribution of improvements across targets (box/violin)
* improvement vs target island size

### B) All phenotypes

Do the same for wing length and tarsus length.
This gives you a clean “method generalizes across traits” story.

---

## 3) Use island-Shapley to do real “training population optimization” comparisons

There’s a whole genomics literature on **training population optimization** (TPO)—CDmean/PEVmean criteria, GA selection, etc. It’s directly aligned with your thesis theme. ([PMC][2])

Add a small baseline suite per target island:

1. **Shapley island selection** (yours)
2. **Relatedness / similarity selection** (no target labels): pick islands/individuals closest to target by GRM/PCA distance
3. **TPO criteria** (classic): CDmean / PEVmean style selection (genomics standard) ([PMC][2])
4. **GA-based training population selection** (if you want one “strong” classical optimizer baseline): Akdemir et al. style genetic algorithm reliability optimization ([Springer Nature Link][3])

That set makes your thesis feel grounded: “game-theoretic valuation vs established genomic TPO.”

---

## 4) Go deeper on interpretation (this is where island-wise Shapley shines)

Once you have Shapley values for all target islands, build:

### A) Island → island contribution matrix

For each target (j), you have (\phi_{k\to j}) for each source island (k).
Plot a heatmap and cluster it.
This is your “dispersal / proximity” narrative vehicle.

### B) Correlate contributions with distance

Test whether high (\phi_{k\to j}) tends to come from:

* geographically close islands, and/or
* genetically close islands (GRM/PCA distance)

Even a simple scatter + regression line per target island tells a strong story.

### C) Compare two rankings explicitly

You already compute:

* **total**: (\phi_k)
* **mean per individual**: (\phi_k / n_k)

Make “add/remove” curves for both ranking modes. This answers the “big islands dominate vs informative islands dominate” question cleanly.

---

## 5) Extend the method beyond “remove islands”

Two easy, thesis-relevant extensions:

### A) Use Shapley values as **weights**, not just selection

Instead of removing islands, train ridge with island weights:

* weight each individual from island (k) by (w_k = \max(\phi_k, 0)) or a soft transform
  This often preserves diversity while still downweighting harmful islands.

### B) Hierarchical selection: island → individual

Now that ridge fits fast, you can afford a second stage:

* select top islands via island Shapley
* within those islands, do individual-level selection using fast ridge leave-one-out / influence-style updates (feasible for ridge)

This gets you closer to the original Data Shapley “per datapoint” spirit without deep-net cost.

---

## 6) Keep one “meta” lesson in the thesis

A nice framing is:

* In ML data valuation benchmarks, no single method wins everywhere; matching the method to the downstream goal matters. ([arXiv][4])
  Your work supports that: TracIn struggled under domain shift + representation learning noise, while island-wise Shapley with a stable model (ridge) produces consistent improvements.

---

## Suggested next 2-week plan

1. **Run all islands for body mass** with ridge + cal/test separation (50 permutations).
2. Add **size-matched random-individual baseline**.
3. Produce:

   * per-target best improvement bar plot
   * island→island heatmap
   * total vs mean ranking comparison
4. Then repeat for **wing + tarsus** (even with fewer permutations initially, like 20).

# In-run data shapley

This Wang et al. “**Data Shapley in One Training Run**” paper is essentially: *“make Shapley-style data valuation feasible for deep nets by doing it during training, not by retraining on many subsets.”*  ([proceedings.iclr.cc][1])

## The idea in the paper

### 1) Redefine “Data Shapley” for a specific training run

Classic Data Shapley (Ghorbani & Zou 2019) defines (U(S)) as: train on subset (S) → evaluate → Shapley over subsets (retraining-heavy).
This paper instead defines a **local utility per SGD step**: “how much did this update reduce validation loss?” and then **sums Shapley contributions over training steps** (“In-Run Data Shapley”). 

### 2) First-order version becomes a simple closed-form (and matches TracIn-Ideal)

Using a first-orde, the per-example contribution at step (t) is:
[
\phi(z) \approx -\eta_t ,\nabla\ell(w_t, z_{val})\cdot \nabla\ell(w_t, z)
]
and you accumulate this across steps whenever (z) is used. 
They explicitly note this coincides with **TracIN-Ideal** (but they make it scalable). ([proceedings.neurips.cc][2])

### 3) The “ghost dot-product” trick makes it fast

The bottleneck is per-sample ngineering contribution is computing **all needed gradient dot-products inside normal backprop** without explicitly forming per-sample gradient vectors (“ghost dot-product”). That’s why they can claim near “regular training” speed for first-order.  ([proceedings.iclr.cc][1])

They also derive a second-order variant with a Hessian interaction term, but empirically first-order already ranks very well. 

## Main empirical results (what they sg (Pile → GPT-2 / Pythia pilot):

* They find a nontrivial fraction of training corpora get **negative value** (they report ~16% in one setting), and removing negative-valued data **improves convergence** (fewer iterations to reach a target loss) and improves performance. ontributions are **stage-dependent**: general corpora help early, domain-specific data dominates later. 
* Runtime: first-order “ghost” implementation runs close to regular training; second-order is ~2× slower; naive per-sample methods are much slower. 

They also clearly flag limitations:ata available during training; efficient implementation is derived for SGD and extending to Adam is nontrivial. teresting sidenote: there is already follow-up work in 2026 specifically tackling Adam-aware in-run Shapley.) ([arXiv][3])

## Could this fit, but with an important reality check

For **your ridge + island-level TMC Shapley**, this paper is *not* the right tool. It’s designed for **iterative gradient training** ution during SGD. 

Where it *does* fit nicely is as a **deep-learning counterpart** to your ridge/TMC results:

* Use **target_cal** as the validation set.
* Train your MLP with SGD.
* Compute **first-order in-run Shapley** (which is essentially TracIn-Ideal) online across steps, then **aggregate scores per training individual**, and optionally sum per island.
* Then do the same “remove low-value → test corr impet_test**.

This is basically “TracIn, but done continuously over the whole run (not just checkpoints) and in principle not restricted to last-layer.” 

### Why it might work better than your earlier TracIn attempt

Your earlier TracIn issues were heavily about: checkpoint choice, last-layer approximations, and late-training overfit to source. In-run Shapley naturally gives you:

* **stage-resolved values** (you can restrict to early training steps where target_cal improves),
* and (if you implement more than last-layer) a better reflection of r. 

### Implementation burden (in your repo)

* **Full “ghost dot-product”** is nontrivial engineering (they reuse internal backprop signals; they even released tooling around it). ([GitHub][4])
* But for your dataset scale, you can do a **simple first-order implementation** without ghost tricks:

  * compute (g_{val} = \nabla \ell(\text{target_cal minibatch}))
  * compute per-example gradieatch (via `functorch.vmap` or your existing last2layers formulas)
  * accumulate dot products
    This is more compute than ridge, but feasible as a “methods comparison chapter”.

## What I’d recommend, given your current strong ridge/TMC results

1. **Make ridge/TMC Shapley your main training-set optimization contribution** (it’s already giving clean gains and interpretable island values).
2. If you want one deep-learning experiment for completeness:

   * implement **first-order in-run Shapley / TracIn-Ideal**, but **only score using early training window** (up to the epoch where target_cal loss is minimal), and compare against:

     * random removal
     * your ridge/TMC island removal
3. Only attempt “ghost dot-product” if you really want a scalable/clean “ICLR’25 method implemented” story—otherwise it’s too much engineering relative to the marginal thesis benefit.

If you tell me whether your MLP training uses **SGD or Adam** in your final setup, I can suggest the simplest in-run experiment that stays faithful to the paper’s assumptions (and whether it’s worth considering the 2026 Adam-aware follow-up).

[1]: https://proceedings.iclr.cc/paper_files/paper/2025/file/20fdaf67581e6d7157376d1ed584040a-Paper-Conference.pdf?utm_source=chatgpt.com "DATA SHAPLEY IN ONE TRAINING RUN"
[2]: https://proceedings.neurips.cc/paper/2020/file/e6385d39ec9394f2f3a354d9d2b88eec-Paper.pdf?utm_source=chatgpt.com "Estimating Training Data Influence by Tracing Gradient Descent"
[3]: https://arxiv.org/abs/2602.00329?utm_source=chatgpt.com "In-Run Data Shapley for Adam Optimizer"
[4]: https://github.com/Jiachen-T-Wang/GhostSuite?utm_source=chatgpt.com "Jiachen-T-Wang/GhostSuite"


# Overfitting

Let $y\in\mathbb{R}^{n}$ be calibration phenotypes ($n=25$), and for each candidate training subset $S$ (size 1000), let $\hat y_S$ be predictions on those same 25 individuals.

You optimize
$$
\hat S=\arg\max_S r_S,\quad
r_S=\mathrm{corr}(y,\hat y_S).
$$

For fixed $n_{\text{train}}=1000$, dividing by $n_{\text{individuals}}$ is constant, so GA is effectively maximizing $r_S$ only.

## Why overfitting appears mathematically

For each $S$, decompose
$$
r_S = \rho_S + \varepsilon_S,
$$
where:
- $\rho_S$: true correlation on new target data,
- $\varepsilon_S$: sampling noise from using only 25 calibration points.

You pick the maximum over many candidates:
$$
r_{\hat S}=\max_S(\rho_S+\varepsilon_S).
$$
Even if $\rho_S$ are moderate, $\max_S \varepsilon_S$ is positive and large (winner’s curse).

### Extreme-value effect
If $M$ candidates are evaluated, roughly
$$
\mathbb E[\max \varepsilon] \propto \sigma \sqrt{2\log M}.
$$
So optimism grows with search size $M$.

With correlation, Fisher transform $z=\operatorname{atanh}(r)$ has
$$
\mathrm{sd}(z)\approx \frac{1}{\sqrt{n-3}}=\frac{1}{\sqrt{22}}\approx 0.213
$$
for $n=25$.  
If effective $M$ is in the thousands, $\sqrt{2\log M}$ is ~3.5–4.3, so max-noise in $z$-space can be substantial, inflating selected $r$.

## Geometric view (same conclusion)

Correlation is cosine between centered vectors:
$$
r_S=\cos\angle(\tilde y,\widetilde{\hat y}_S).
$$
You have many candidate vectors $\widetilde{\hat y}_S\in\mathbb{R}^{25}$.  
Maximizing over many directions makes the best angle very small, so cosine approaches 1 on calibration, even if generalization is poor.

---

So $r_{\text{cal}}\approx 0.99$ with low $r_{\text{test}}$ is exactly what you expect when optimizing on a tiny calibration set with many subset evaluations.